# 07a — Test evaluation: Siamese concat (Focal + Dice, 40 epochs) and baseline

In [ ]:
BRANCH = "main"

%cd /tmp
!rm -rf damage-detection
!git clone -b {BRANCH} https://github.com/azrairemd/damage-detection.git
%cd damage-detection
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
import json
import torch

from src.data.dataset import XBDDataset
from src.evaluate import evaluate, load_model
from src.models.baseline_unet import BaselineUNet
from src.models.siamese_unet import SiameseUNet
from src.utils.config import load_config

In [ ]:
FINAL_PATH = "/kaggle/input/notebooks/azra41221/06c-train-siamese-fd/outputs/siamese_concat_fd40/best.pt"
BASELINE_PATH = "/kaggle/input/datasets/azra41221/baseline-checkpoint/best.pt"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
final_model = load_model(SiameseUNet(encoder="resnet34", fusion="concat"), FINAL_PATH, device)
baseline_model = load_model(BaselineUNet(encoder="resnet34"), BASELINE_PATH, device)

## 1. Sanity check on the validation set

In [ ]:
config = load_config("configs/siamese_concat_fd40.yaml")
data = config["data"]

val_ds = XBDDataset(data["val_split"], data["xbd_root"], data["mask_root"], train=False)
print("Validation images:", len(val_ds))

val_overall, _, _ = evaluate(final_model, val_ds, device)

print("Validation score:", round(val_overall["score"], 4), " (during training: 0.6285)")
print("Localization F1: ", round(val_overall["f1_loc"], 4), " (during training: 0.8006)")
print("Damage F1:       ", round(val_overall["f1_dmg"], 4), " (during training: 0.5547)")

**Sanity check** The loaded Focal + Dice model gives exactly the validation scores logged at epoch 12 during training (0.6285, loc 0.8006, dmg 0.5547).

## 2. Test set evaluation (run once)

In [ ]:
test_ds = XBDDataset(data["test_split"] if "test_split" in data else "splits/test.csv",
                     data["xbd_root"], data["mask_root"], train=False)
print("Test images:", len(test_ds))

results = {}
for name, model in [("siamese_focal_dice", final_model), ("baseline", baseline_model)]:
    overall, per_type, confusion = evaluate(model, test_ds, device)
    results[name] = {"overall": overall, "per_type": per_type, "confusion": confusion.tolist()}
    print(f"{name:20s} test score {overall['score']:.4f}  loc {overall['f1_loc']:.4f}  dmg {overall['f1_dmg']:.4f}")

with open("/kaggle/working/test_results_fd.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved to /kaggle/working/test_results_fd.json")